In [ ]:
import os, sys
# import utils.data_processing as dp
# import utils.pair_methods as pm
import scipy
from datetime import datetime
import re
from collections import defaultdict, OrderedDict
#torch
import torch
from torchvision.models import alexnet
from torchvision import transforms as T
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
import scipy.optimize as optimize

# from dwave.samplers import SimulatedAnnealingSampler

# import dimod
from scipy.linalg import block_diag
from scipy.spatial.distance import cdist
from munkres import Munkres, print_matrix
# Qiskit
import qiskit_optimization as qo
from qiskit_optimization import QuadraticProgram

# quimb



# Problem modelling imports
from docplex.mp.model import Model

# Qiskit imports
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms.utils import algorithm_globals
from qiskit.primitives import Sampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer, CplexOptimizer
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.problems.variable import VarType
from qiskit_optimization.converters.quadratic_program_to_qubo import QuadraticProgramToQubo
from qiskit_optimization.translators import from_docplex_mp


In [ ]:
#search for and find the files
def path_joiner(image_name, root_dir = None): #recursive search for the image
    if root_dir is None:
        root_dir = os.getcwd()
    for dirpath , _, filenames in os.walk(root_dir):
        if image_name in filenames:
            return os.path.join(dirpath, image_name)
#usage
mat_files = ['Cars_006a.mat','Cars_007a.mat','Cars_008b.mat']
paths = []
for fi in mat_files:
    paths.append(path_joiner(fi))

In [ ]:
#Keypoints extraction
def keypoint(image_path, max_points=None): # extract a desired number of keypoints from the images
    keypoints1 = scipy.io.loadmat(image_path)["pts_coord"]
    if max_points is not None:
        keypoints1 = keypoints1[:, :max_points]
    return keypoints1
def keypoints_dict(image_names: list, max_points: int):
    keypoints = {}
    for image_name in image_names:
        base_name = os.path.splitext(image_name)[0]
        full_path = path_joiner(image_name)
        if full_path:
            kps = keypoint(full_path, max_points)
            keypoints[base_name] = kps
        else:
            print(f"[Warning] image not found: {image_name}")
    return keypoints
points_dic = keypoints_dict(mat_files,max_points=3)
print(f'This is the keypoints dict {points_dic}')

In [ ]:
#Feature extraction using AlexNet

def alexnet_feature_extractor(layer= 'conv4'):
    model = alexnet(pretrained=True)
    model.eval()
    if layer == 'conv4':
        return torch.nn.Sequential(*list(model.features)[:10])
    elif layer == 'conv5':
        return torch.nn.Sequential(*list(model.features)[:12])
    else:
        raise ValueError("Invalid layer. Choose 'conv4' or 'conv5'.")
def extract_features(keypoints_dict, patch_size=64, layer='conv4'):
    feature_extractor = alexnet_feature_extractor(layer)
    transform = T.Compose([
        T.Resize((227, 227)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feature_extractor.to(device)
    features = {}

    for image_name, keypoints in keypoints_dict.items():
        img_path = path_joiner(image_name + '.png')
        img = Image.open(img_path).convert('RGB')
        img_np = np.array(img)
        feature_list = []

        for (x, y) in keypoints.T:
            x, y = int(round(x)), int(round(y))
            x1 = max(0, x - patch_size // 2)
            y1 = max(0, y - patch_size // 2)
            x2 = min(img.width, x + patch_size // 2)
            y2 = min(img.height, y + patch_size // 2)

            patch = img.crop((x1, y1, x2, y2))
            patch_tensor = transform(patch).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = feature_extractor(patch_tensor)
                feat = feat.mean(dim=[2, 3]) # to flatten the output dimensions
            feature_list.append(feat.squeeze().cpu().numpy())

        features[image_name] = np.stack(feature_list)
    return features
features = extract_features(points_dic)
print(features)

In [ ]:
#features have been extracted, now we need to calculate the distance matrix
def pairwise_permutations(features_dict, pm_size: int) -> dict: # compute the P_ij
    P = {}
    image_list = list(features_dict.keys())

    for i in range(len(image_list)):
        key0= f"P{i+1}{i+1}"
        P[key0] = np.eye(pm_size)
        for j in range(i + 1, len(image_list)):
            img1 = image_list[i]
            img2 = image_list[j]

            feats1 = features_dict[img1]
            feats2 = features_dict[img2]
            similarity = cosine_similarity(feats1, feats2)
            cost_mat = (1 - similarity)
            row_ind, col_ind = linear_sum_assignment(cost_mat)
            n = len(row_ind)
            perm_matrix = np.zeros((n, n), dtype=int)
            perm_matrix[row_ind, col_ind] = 1
            key1 = f"P{i+1}{j+1}"
            P[key1] = perm_matrix
            key2 = f"P{j+1}{i+1}"
            P[key2] = perm_matrix.T
    return P
P = pairwise_permutations(features, 3)
print(len(P))
print(P)

In [ ]:
#Qubo formulation using Qiskit-optimization
def qubo_form_maker(P: dict, num_views: int, num_keypoints: int, penalty: float=1.5):
    # without setting the X1 to identity
    mdl = QuadraticProgram("QPS")
    m = num_views
    n = num_keypoints
    N = n * n
    #size of the x vector num_viewsx(num_keypoints^2)
    for i in range(m):
        for j in range(N):
            mdl.binary_var(name=f'x{i}{j}')
    
    #constructing the Q_prime (m.N)x(m.N)
    Q_prime = np.zeros((m*N, m*N))
    for key, p_mat_ij in P.items():
        # extract i and j from key like "P12"
        i = int(key[1]) - 1  # subtract 1 for zero-based indexing
        j = int(key[2]) - 1
        block_ij = np.kron(np.eye(n), p_mat_ij)
        for u in range(N):
            for v in range(N):
                Q_prime[i*N + u, j*N + v] -= block_ij[u, v]
    # constructing the A and b as mentioned in the paper
    A = []
    b = []

    for i in range(m):
        #Row constraints
        for row in range(n):
            constraint = np.zeros(m*N)
            for col in range(n):
                constraint[i*N+row*n+col] = 1
            A.append(constraint)
            b.append(1)
        # Column constraint
        for col in range(n):
            constraint = np.zeros(m*N)
            for row in range(n):
                idx = i * N + row * n + col
                constraint[idx] = 1
            A.append(constraint)
            b.append(1)
    A = np.array(A)
    b = np.array(b)

    #constructing Q and s
    Q = Q_prime + penalty * A.T @ A
    s = -2 * penalty * A.T @ b
    # putting things together
    
    
    mdl.minimize(linear=s, quadratic=Q)
    return mdl


#Run
qubo_model = qubo_form_maker(P, 3, 3, 1.5)
print(qubo_model.prettyprint())
print(type(qubo_model))

In [ ]:
import copy
# Using warm starting 
def relax_problem(problem) -> QuadraticProgram:
    """Change all variables to continuous."""
    relaxed_problem = copy.deepcopy(problem)
    for variable in relaxed_problem.variables:
        variable.vartype = VarType.CONTINUOUS

    return relaxed_problem

In [ ]:
# run
qubo_model_relaxed = relax_problem(qubo_model)
print(qubo_model_relaxed.prettyprint())

In [ ]:
from qiskit_optimization.algorithms import WarmStartQAOAOptimizer, SlsqpOptimizer
from qiskit.primitives import Estimator



In [ ]:
# 1) Create any continuous presolver

slsqp = SlsqpOptimizer()          # Pure SciPy
relaxed_solution = slsqp.solve(qubo_model_relaxed)
c_stars = relaxed_solution.x      # numpy array in [0,1]
print(c_stars)


In [ ]:
algorithm_globals.random_seed = 12345
# qaoa_mes = QAOA(sampler=Sampler(), optimizer=COBYLA(), initial_point=[0.0, 1.0])
# exact_mes = NumPyMinimumEigensolver()
# qaoa = MinimumEigenOptimizer(qaoa_mes)
# qaoa_result = qaoa.solve(qubo_model)
# print(qaoa_result.prettyprint())

In [ ]:
# creating warm_starting QAOA circuit
from qiskit import QuantumCircuit

thetas = [2 * np.arcsin(np.sqrt(c_star)) for c_star in c_stars]

init_qc = QuantumCircuit(27)
for idx, theta in enumerate(thetas):
    init_qc.ry(theta, idx)

init_qc.draw()

In [ ]:
#create the mixer
from qiskit.circuit import Parameter

beta = Parameter("β")

ws_mixer = QuantumCircuit(27)
for idx, theta in enumerate(thetas):
    ws_mixer.ry(-theta, idx)
    ws_mixer.rz(-2 * beta, idx)
    ws_mixer.ry(theta, idx)

ws_mixer.draw()

In [ ]:
ws_qaoa_mes = QAOA(
    sampler=Sampler(),
    optimizer=COBYLA(),
    initial_state=init_qc,
    mixer=ws_mixer,
    initial_point=[0.0, 1.0],
)
ws_qaoa = MinimumEigenOptimizer(ws_qaoa_mes)

In [ ]:
ws_qaoa_result = ws_qaoa.solve(qubo_model)
print(ws_qaoa_result.prettyprint())

In [ ]:
def qubo_to_ising(model : qo.problems.quadratic_program.QuadraticProgram):
    return qo.translators.to_ising(model)

ising_model, offset = qubo_to_ising(qubo_model_relaxed)
print(ising_model)
print(offset)

In [ ]:
print(type(ising_model))

In [ ]:
# from qiskit_optimization.translators import to_ising
# from qiskit.quantum_info import SparsePauliOp

# def paulisumop_to_quimb_terms(pauli_op: SparsePauliOp):
#     """
#     Converts Qiskit's SparsePauliOp to quimb-style Ising dict:
#     keys = tuple of qubit indices, values = coefficients
#     """
#     terms = {}
#     for pauli, coeff in zip(pauli_op.paulis, pauli_op.coeffs):
#         z_indices = [i for i, p in enumerate(pauli.to_label()) if p == 'Z']
#         if len(z_indices) > 0:
#             terms[tuple(z_indices)] = coeff.real
#     return terms